# Part 1: Pull MORF Schedule from Website
MORF Website: https://morfracing.net/wpmocha/schedule

In [28]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import os
from datetime import datetime

In [4]:
# URL of the MORF schedule page with 100 entries shown
URL = "https://morfracing.net/wpmocha/schedule"

# Fetch page
response = requests.get(URL)
response.raise_for_status()  # stops if status != 200

# Parse the HTML
soup = BeautifulSoup(response.text, "html.parser")

# Find the schedule table (DataTables)
table = soup.find("table")

In [24]:
# List to store rows
rows_data = []

# Parse table body
if table:
    tbody = table.find("tbody")
    if tbody:
        for row in tbody.find_all("tr"):
            cols = row.find_all("td")
            # Ensure expected columns exist
            if len(cols) >= 7:
                rows_data.append({
                    "Day": cols[0].get_text(strip=True),
                    "Date": cols[1].get_text(strip=True),
                    "Start Time": cols[2].get_text(strip=True),
                    "Course": cols[3].get_text(strip=True),
                    "Event - Results": cols[4].get_text(strip=True),
                    "Series": cols[5].get_text(strip=True),
                })

# Create DataFrame
df = pd.DataFrame(rows_data)

df

,Day,Date,Start Time,Course,Event - Results,Series
0,Wed,05/13/26,06:40:00 PM,SA7,Beer Can - Tuneup Race,Beer
1,Sun,05/17/26,11:00:00 AM,SA7,Performance Series - Race 1 (Olympic),Perf
2,Wed,05/20/26,06:40:00 PM,SA7,Beer Can - Race 1,Beer
3,Sat,05/23/26,11:00:00 AM,SA7,Performance Series - Race 2 (Btrap) Weathermar...,Perf
4,,,,SA7,Performance Series - Race 3 (W-L) Weathermark ...,Perf
...,...,...,...,...,...,...
62,,,,SA7,MORF Open - Race 2 Area-III,Opat
63,,,,SA7,MORF Open - Race 3,Open
64,,,,SA7,MORF Open - Race 3 Area-III,Opat
65,,,,,Boat of the Year,Boty


## Clean Data

In [29]:
# Replace empty strings with NaN so pandas recognizes them as missing
df["Day"] = df["Day"].replace("", pd.NA)
df["Date"] = df["Date"].replace("", pd.NA)

# Forward fill Day and Date only
df["Day"] = df["Day"].ffill()
df["Date"] = df["Date"].ffill()

# Treat empty strings as missing
df["Course"] = df["Course"].replace("", pd.NA)

# Drop rows where Course is blank
df = df.dropna(subset=["Course"]).reset_index(drop=True)

# Drop rows where 'Event - Results' contains 'Beer Can'
mask_beer_can = df["Event - Results"].str.contains(
    "Beer Can", case=False, na=False
)

df = df[~mask_beer_can].reset_index(drop=True)

# Ensure consistent spacing
df["Event - Results"] = df["Event - Results"].str.strip()

# Split on the FIRST hyphen only
split_cols = df["Event - Results"].str.split(" - ", n=1, expand=True)

# Create new column
df["Series Type"] = split_cols[0].str.strip()

df["Series Type"] = (
    df["Series Type"]
    .str.replace(r"\bSeries\b", "", regex=True)
    .str.replace(r"\s{2,}", " ", regex=True)
    .str.strip()
)

# Default value (optional but helpful)
df["Race Type"] = np.nan

# Condition 1: Performance → Buoy
mask_buoy = df["Series Type"].str.strip().eq("Performance")

df.loc[mask_buoy, "Race Type"] = "Buoy"

# Condition 2: Long Distance + contains 'to' or 'from' → port-to-port
mask_ld = df["Series Type"].str.strip().eq("Long Distance")
mask_port = df["Event - Results"].str.contains(r"\b(to|from)\b", case=False, na=False)

df.loc[mask_ld & mask_port, "Race Type"] = "Distance (port-to-port)"

# Condition 3: Long Distance + does NOT contain 'to' or 'from' → nearshore
df.loc[mask_ld & ~mask_port, "Race Type"] = "Distance (nearshore)"

display(len(df))
df.head()

C:\Users\Nicholas\AppData\Local\Temp\ipykernel_23772\1589310661.py:48: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_port = df["Event - Results"].str.contains(r"\b(to|from)\b", case=False, na=False)


43

,Day,Date,Start Time,Course,Event - Results,Series,Series Type,Race Type
0,Sun,05/17/26,11:00:00 AM,SA7,Performance Series - Race 1 (Olympic),Perf,Performance,Buoy
1,Sat,05/23/26,11:00:00 AM,SA7,Performance Series - Race 2 (Btrap) Weathermar...,Perf,Performance,Buoy
2,Sat,05/23/26,,SA7,Performance Series - Race 3 (W-L) Weathermark ...,Perf,Performance,Buoy
3,Sat,05/30/26,11:00:00 AM,SA7,Performance Series - Race 4 (Trapezoid),Perf,Performance,Buoy
4,Sun,05/31/26,10:00:00 AM,SA7,Long Distance Series - Race 1 (Zimmer),Long,Long Distance,Distance (nearshore)


## Export Data

In [30]:
# Create folder if it doesn't exist
folder_name = "morf_schedule"
os.makedirs(folder_name, exist_ok=True)

# Current year
year = datetime.now().year

# Build full file path with year in filename
file_path = os.path.join(folder_name, f"morf_schedule_{year}.csv")

# Save CSV into that folder
df.to_csv(file_path, index=False)

print(f"Saved file to: {file_path}")

Saved file to: morf_schedule\morf_schedule_2026.csv
